In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [14]:
import os
import re
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# =========================
# PATHS
# =========================

BASE_PATHS = {
    "NMF": Path("/home/moshtasa/Research/phd-svd-recsys/Mobile/0405/0405_NMF/result/NMF_run/NMF_bar_figures"),
    "SVD": Path("/home/moshtasa/Research/phd-svd-recsys/Mobile/0405/0405_SVD/result/SVD_bar_figures"),
}

# =========================
# PARSE TXT
# =========================

def parse_txt(file_path):
    data = {}
    current_k = None

    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()

            if line.startswith("k ="):
                current_k = int(line.split("=")[1].strip())
                data[current_k] = {}

            elif ":" in line and current_k is not None:
                match = re.match(r"(.*?)\:\s*([0-9.]+)", line)
                if match:
                    label = match.group(1).strip()
                    value = float(match.group(2))
                    data[current_k][label] = value

    return data

# =========================
# COLORS
# =========================

def generate_colors(n_list):
    cmap = plt.cm.get_cmap('tab20', len(n_list))
    return {n: cmap(i) for i, n in enumerate(n_list)}

# =========================
# EXTRACT GENRE
# =========================

def extract_genre(folder_name):
    parts = folder_name.lower().split("_")
    return parts[1] if len(parts) > 1 else None

# =========================
# BUILD MAPS
# =========================

nmf_map = {
    extract_genre(f): f
    for f in os.listdir(BASE_PATHS["NMF"])
    if (BASE_PATHS["NMF"] / f).is_dir()
}

svd_map = {
    extract_genre(f): f
    for f in os.listdir(BASE_PATHS["SVD"])
    if (BASE_PATHS["SVD"] / f).is_dir()
}

common_genres = sorted(set(nmf_map.keys()) & set(svd_map.keys()))

print("Detected genres:", common_genres)

# =========================
# MAIN LOOP
# =========================

for genre_key in common_genres:

    genre = genre_key.capitalize()

    nmf_folder = nmf_map[genre_key]
    svd_folder = svd_map[genre_key]

    paths = {
        "NMF": BASE_PATHS["NMF"] / nmf_folder,
        "SVD": BASE_PATHS["SVD"] / svd_folder,
    }

    parsed_data = {}

    for method, folder_path in paths.items():
        txt_files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]
        if not txt_files:
            print(f"No txt in {folder_path}")
            continue

        txt_path = folder_path / txt_files[0]
        parsed_data[method] = parse_txt(txt_path)

    if len(parsed_data) < 2:
        print(f"Skipping {genre_key}")
        continue

    ks = [k for k in sorted(parsed_data["NMF"].keys()) if k in [15, 25, 35]]
    if not ks:
        continue

    n_types = sorted(set(
        n
        for method in parsed_data
        for k in ks
        for n in parsed_data[method][k].keys()
    ), key=lambda x: (x != "original", int(x.replace("u", "")) if x != "original" else -1))

    color_map = generate_colors(n_types)

    fig, axes = plt.subplots(1, 2, figsize=(20, 8), sharey=True)

    legend_handles = []
    legend_labels = []

    for ax, method in zip(axes, ["NMF", "SVD"]):

        data = parsed_data[method]

        x = np.arange(len(ks)) * 1.3
        width = 1.1 / len(n_types)

        for i, n in enumerate(n_types):
            vals = [data[k].get(n, 0) for k in ks]

            bars = ax.bar(
                x + (i - len(n_types)/2) * width + width/2,
                vals,
                width=width,
                color=color_map[n],
                edgecolor="black",
                linewidth=1.5,
                label=n
            )

            # collect legend only once
            if method == "NMF":
                legend_handles.append(bars[0])
                legend_labels.append(n)

        ax.spines["top"].set_visible(True)
        ax.spines["right"].set_visible(True)
        ax.yaxis.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)

        ax.set_title(f"{method} - {genre}", fontsize=28, fontweight="bold", pad=10)

        ax.set_xticks(x)
        ax.set_xticklabels(ks, fontsize=24)
        ax.tick_params(axis='y', labelsize=24)

        ax.set_ylim(0, 40)
        ax.set_xlabel("k (recommendation size)", fontsize=26, fontweight="bold", labelpad=10)

    axes[0].set_ylabel("Avg count per user", fontsize=26, fontweight="bold", labelpad=10)

    # =========================
    # COMPACT LEGEND (FIXED)
    # =========================
    fig.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        bbox_to_anchor=(0.35, 1.1),   # ✅ no wasted top space
        fontsize=26,
        ncol=min(len(legend_labels), 8),
        handlelength=1.3,
        handletextpad=0.4,
        columnspacing=0.7
    )

    # tighter layout to remove empty space
    plt.tight_layout(rect=[0, 0, 1, 0.93])

    filename = f"comparision - {genre}.png"

    plt.savefig(paths["NMF"] / filename, dpi=300, bbox_inches="tight")
    plt.savefig(paths["SVD"] / filename, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Saved: {filename}")

print("DONE ✅")

Detected genres: ['shopping']


/tmp/ipykernel_1277419/1603471402.py:46: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('tab20', len(n_list))


Saved: comparision - Shopping.png
DONE ✅


In [17]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

# =========================
# PATHS
# =========================

# --- SVD ---
SVD_BASE = "/home/moshtasa/Research/phd-svd-recsys/Mobile/0405/0405_SVD/result"
SVD_BI_DIR = Path(SVD_BASE) / "bi"
SVD_MU_DIR = Path(SVD_BASE) / "mio"
SVD_MERGED_DIR = Path(SVD_BASE) / "merged"
SVD_FIG_DIR = Path(SVD_BASE) / "SVD_details_figures"

# --- NMF ---
NMF_BASE = "/home/moshtasa/Research/phd-svd-recsys/Mobile/0405/0405_NMF/result/NMF_run"
NMF_BI_DIR = Path(NMF_BASE) / "bi"
NMF_MU_DIR = Path(NMF_BASE) / "mio"
NMF_MERGED_DIR = Path(NMF_BASE) / "merged"
NMF_FIG_DIR = Path(NMF_BASE) / "NMF_details_figures"

# Ensure dirs exist
SVD_MERGED_DIR.mkdir(parents=True, exist_ok=True)
NMF_MERGED_DIR.mkdir(parents=True, exist_ok=True)
SVD_FIG_DIR.mkdir(parents=True, exist_ok=True)
NMF_FIG_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# STYLE
# =========================

plt.style.use("seaborn-v0_8-pastel")

TARGET_COLOR = "#5B3A29"
TARGET_LINEWIDTH = 3.5
NON_TARGET_LINEWIDTH = 1.5

PASTEL_COLORS = [
    "#AEC6CF","#FFD1DC","#B5EAD7","#FFDAC1","#C3B1E1",
    "#F3E5AB","#FFB7B2","#B0E0E6","#D5AAFF","#FFFACD",
    "#E0BBE4","#957DAD","#D4E157","#80CBC4","#F48FB1",
]

# =========================
# HELPERS
# =========================

def rename_to_category(df):
    for col in ["genre", "genre_g1", "category"]:
        if col in df.columns:
            return df.rename(columns={col: "category"})
    return df

# =========================
# LOAD + MERGE
# =========================

def load_and_merge(bi_dir, mu_dir, bi_pattern, mu_pattern):

    bi_map = {}
    mu_map = {}

    for f in Path(bi_dir).glob("*.csv"):
        m = bi_pattern.match(f.name)
        if m:
            bi_map[(m.group(1), int(m.group(2)))] = f

    for f in Path(mu_dir).glob("*.csv"):
        m = mu_pattern.match(f.name)
        if m:
            mu_map[(m.group(1), int(m.group(2)))] = f

    merged_data = {}

    for key in sorted(bi_map.keys()):
        if key not in mu_map:
            continue

        df_bi = rename_to_category(pd.read_csv(bi_map[key]))
        df_mu = rename_to_category(pd.read_csv(mu_map[key]))

        merge_cols = ["category"]
        if "is_target" in df_bi.columns and "is_target" in df_mu.columns:
            merge_cols.append("is_target")

        df = pd.merge(df_mu, df_bi, on=merge_cols, how="outer")
        merged_data[key] = df

    return merged_data

# =========================
# LOAD DATA
# =========================

SVD_BI_PATTERN = re.compile(r"^f_(.+?)_(\d+)u_.*_bi_.*\.csv$")
SVD_MU_PATTERN = re.compile(r"^f_(.+?)_(\d+)u_.*_mu_.*\.csv$")

NMF_BI_PATTERN = re.compile(r"^f_(.+?)_(\d+)u_.*_bi_.*\.csv$")
NMF_MU_PATTERN = re.compile(r"^f_(.+?)_(\d+)u_.*_mu_.*\.csv$")

svd_data = load_and_merge(SVD_BI_DIR, SVD_MU_DIR, SVD_BI_PATTERN, SVD_MU_PATTERN)
nmf_data = load_and_merge(NMF_BI_DIR, NMF_MU_DIR, NMF_BI_PATTERN, NMF_MU_PATTERN)

# =========================
# GROUP
# =========================

svd_groups = defaultdict(dict)
nmf_groups = defaultdict(dict)

for (k, n), df in svd_data.items():
    svd_groups[k][n] = df

for (k, n), df in nmf_data.items():
    nmf_groups[k][n] = df

# =========================
# MAIN LOOP
# =========================

for genre_key in sorted(nmf_groups.keys()):

    if genre_key not in svd_groups:
        continue

    nmf_dict = nmf_groups[genre_key]
    svd_dict = svd_groups[genre_key]

    sorted_ns = sorted(set(nmf_dict.keys()) & set(svd_dict.keys()))
    if not sorted_ns:
        continue

    sample_df = nmf_dict[sorted_ns[0]]
    target_rows = sample_df[sample_df["is_target"] == "target"]
    if target_rows.empty:
        continue

    target_name = target_rows["category"].iloc[0]

    def build_series(n_dict):
        all_categories = set()
        for n in sorted_ns:
            all_categories.update(n_dict[n]["category"].tolist())
        all_categories = sorted(all_categories)

        series = {}
        for g in all_categories:
            pts = []
            for n in sorted_ns:
                df = n_dict[n]
                row = df[df["category"] == g]
                if not row.empty:
                    val = row["avg_bi"].values[0] + row["global_mean_mu"].values[0]
                    pts.append((n, val))
            series[g] = pts

        return series, all_categories

    nmf_series, all_categories = build_series(nmf_dict)
    svd_series, _ = build_series(svd_dict)

    non_target = [g for g in all_categories if g != target_name]

    color_map = {g: PASTEL_COLORS[i % len(PASTEL_COLORS)] for i, g in enumerate(non_target)}
    color_map[target_name] = TARGET_COLOR

    positions = list(range(len(sorted_ns)))
    n_to_pos = {n: p for n, p in zip(sorted_ns, positions)}

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True, constrained_layout=True)

    legend_handles = {}

    for ax, series in zip(axes, [nmf_series, svd_series]):

        for g in non_target:
            pts = series.get(g, [])
            if pts:
                xs = [n_to_pos[x] for x, _ in pts]
                ys = [y for _, y in pts]
                line, = ax.plot(xs, ys, marker="o",
                                color=color_map[g],
                                linewidth=NON_TARGET_LINEWIDTH,
                                markersize=9,
                                alpha=0.8,
                                label=g)
                if g not in legend_handles:
                    legend_handles[g] = line

        pts = series.get(target_name, [])
        if pts:
            xs = [n_to_pos[x] for x, _ in pts]
            ys = [y for _, y in pts]
            line, = ax.plot(xs, ys, marker="o",
                            color=TARGET_COLOR,
                            linewidth=TARGET_LINEWIDTH,
                            markersize=7,
                            label=f"{target_name} (target)",
                            zorder=10)
            legend_handles[target_name] = line

        ax.tick_params(axis='y', labelsize=18)
        ax.tick_params(axis='x', labelsize=20)
        ax.set_xlabel("Number of Fictitious Users (n)", fontsize=22, fontweight="bold")
        ax.set_xticks(positions)
        ax.set_xticklabels([str(n) for n in sorted_ns], rotation=45)
        ax.grid(True, alpha=0.3)

    # ===== TITLES (ONLY CHANGE) =====
    axes[0].set_title(f"NMF - {genre_key}", fontsize=22, fontweight="bold")
    axes[1].set_title(f"SVD - {genre_key}", fontsize=22, fontweight="bold")

    axes[0].set_ylabel(r"$\mu$ + $b_i$", fontweight="bold", fontsize=26)

    fig.legend(
        list(legend_handles.values()),
        list(legend_handles.keys()),
        loc="upper center",
        bbox_to_anchor=(0.5, 1.5),
        fontsize=16,
        ncol=min(len(legend_handles), 6),
        handlelength=1.2,
        handletextpad=0.6,
        columnspacing=0.8
    )

    # =========================
    # SAVE (BOTH FOLDERS)
    # =========================

    filename = "shopping_comparision.png"

    svd_path = SVD_FIG_DIR / filename
    nmf_path = NMF_FIG_DIR / filename

    fig.savefig(svd_path, dpi=150, bbox_inches="tight")
    fig.savefig(nmf_path, dpi=150, bbox_inches="tight")

    plt.close(fig)

    print(f"Saved: {svd_path}")
    print(f"Saved: {nmf_path}")

print("Done!")

Saved: /home/moshtasa/Research/phd-svd-recsys/Mobile/0405/0405_SVD/result/SVD_details_figures/shopping_comparision.png
Saved: /home/moshtasa/Research/phd-svd-recsys/Mobile/0405/0405_NMF/result/NMF_run/NMF_details_figures/shopping_comparision.png
Done!
